In [10]:
import torch
import pandas as pd
from src.configuration.config import set_seed, SEED
from src.configuration.hyperparameter_search import sequential_hyperparameter_search, base_config, search_space
from src.utils.data import sources, au_cols

In [11]:
set_seed(SEED)

In [12]:
df = pd.read_csv("../data/processed/combined.csv")

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameter Tuning
### Testing different Configs with Coordinate Search based on the full model (All eight AUs)

In [14]:
best_config, summary = sequential_hyperparameter_search(
    df=df,
    test_au=au_cols,
    base_config=base_config,
    search_space=search_space,
    sources=sources,
    device=device
)


Best result in current step:
Parameter: window_config
Value: {'window_size': 10, 'stride': 5}
Mean Loss: 0.5449

Best result in current step:
Parameter: learning_rate
Value: 0.003
Mean Loss: 0.5694

Best result in current step:
Parameter: weight_decay
Value: 0.001
Mean Loss: 0.5817

Best result in current step:
Parameter: batch_size
Value: 16
Mean Loss: 0.5817

Best result in current step:
Parameter: dropout
Value: 0.2
Mean Loss: 0.5817


In [15]:
best_config

{'window_size': 10,
 'stride': 5,
 'learning_rate': 0.003,
 'weight_decay': 0.001,
 'batch_size': 16,
 'dropout': 0.2,
 'num_epochs': 25,
 'threshold': 0.5}

In [16]:
summary["param_value_group"] = summary["param_value"].apply(
    lambda x: str(x) if isinstance(x, dict) else x
)

summary = (
    summary
    .groupby(["param_name", "param_value_group"],
             as_index=False,
             sort=False)
    .mean(numeric_only=True)
)

summary["macro_f1"] = (summary["f1_0"] + summary["f1_1"]) / 2

Config is chosen by max. Balanced Accuracy instead of min. Loss -> Min. the loss often leads to prioritizing class 1. Choosing Balanced Accuracy increases the precision_0, recall_0 and f1_0 significantly in later tested hyperparameters, while performance for 1 does not drastically change. BUT Loss increases

In [17]:
summary

,param_name,param_value_group,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_f1
0,window_config,"{'window_size': 10, 'stride': 5}",0.723799,0.544907,0.485714,0.626263,0.314815,0.7750,0.382022,0.692737,0.537380
1,window_config,"{'window_size': 10, 'stride': 10}",0.703012,0.532639,0.468750,0.617647,0.277778,0.7875,0.348837,0.692308,0.520572
2,window_config,"{'window_size': 20, 'stride': 10}",0.737686,0.511343,0.434783,0.603604,0.185185,0.8375,0.259740,0.701571,0.480655
3,window_config,"{'window_size': 20, 'stride': 20}",0.724828,0.498843,0.400000,0.596330,0.185185,0.8125,0.253165,0.687831,0.470498
4,window_config,"{'window_size': 30, 'stride': 15}",0.720652,0.471065,0.318182,0.580357,0.129630,0.8125,0.184211,0.677083,0.430647
5,window_config,"{'window_size': 30, 'stride': 30}",0.733614,0.517361,0.444444,0.607477,0.222222,0.8125,0.296296,0.695187,0.495742
6,learning_rate,0.0001,0.683831,0.454861,0.324324,0.567010,0.222222,0.6875,0.263736,0.621469,0.442603
7,learning_rate,0.0003,0.666366,0.508102,0.423077,0.601852,0.203704,0.8125,0.275000,0.691489,0.483245
8,learning_rate,0.001,0.723799,0.544907,0.485714,0.626263,0.314815,0.7750,0.382022,0.692737,0.537380
9,learning_rate,0.003,0.794119,0.569444,0.512195,0.645161,0.388889,0.7500,0.442105,0.693642,0.567873


# Convert To LaTeX

In [18]:
summary.to_latex(
    "../results/attention_based_mil/hyperparameter_tuning/hyperparameter_config_BalAcc.tex",
    index=False,
    float_format="%.3f"
)